# Mini-Project MP03 — Press Release to Plot

## Industry Comparison: Financial Services and Travel and Hospitality

*CIS 3120 — Programming for Analytics*
*Baruch College, Zicklin School of Business*

---

**Team number:** `16`

**Team members:**
- Financial Services Pipeline Lead: `Chinmoy Chowdhury`
- Travel and Hospitality Pipeline Lead: `[TH Lead Name]`
- Comparison and Visualization Lead (Integrator):

**Submission filename:** `MP03_Notebook_team_16.ipynb`

---

## How to use this starter

1. Make a copy of this notebook and rename it `MP03_Notebook_team_16.ipynb`.
2. Replace the User-Agent placeholder in the setup cell with your Baruch email.
3. Configure your Anthropic API key in Colab Secrets as `ANTHROPIC_API_KEY`.
4. Work through the notebook section by section. Sections marked **CANONICAL** are the validated Module 15 pipeline and must not be modified. Sections marked **TODO** are where your team writes new code.
5. Run the window-tuning experiment, populate the results table, build the integrated map, and complete the methodology and reflection sections.
6. Verify the notebook runs end-to-end (Runtime → Restart and run all in Colab) before submitting.

See `docs/MP03_Assignment.docx` for the full assignment specification.

---

## 1. Setup

Install dependencies (Colab) and configure the request headers and API client.

In [2]:
# Colab installs (silent). The other packages are pre-installed in the Colab base image.
!pip install anthropic folium --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 753.6/753.6 kB 16.0 MB/s eta 0:00:00


In [3]:
import json
import re
import time
from datetime import date, datetime, timedelta

import requests
from bs4 import BeautifulSoup
import folium
import pandas as pd
from anthropic import Anthropic

# ─────────────────────────────────────────────────────────────────────────
# CRITICAL: Replace the placeholder below with your Baruch email.
# Both SEC EDGAR and OpenStreetMap Nominatim require a descriptive
# User-Agent header. Generic agents are rejected with HTTP 403.
# ─────────────────────────────────────────────────────────────────────────
USER_AGENT = "CIS3120 MP03 Team 16 - your.email@baruch.cuny.edu"

REQUEST_HEADERS = {"User-Agent": USER_AGENT}

# ─────────────────────────────────────────────────────────────────────────
# Endpoints and constants
# ─────────────────────────────────────────────────────────────────────────
EDGAR_SEARCH_URL = "https://efts.sec.gov/LATEST/search-index"
NOMINATIM_URL    = "https://nominatim.openstreetmap.org/search"

EDGAR_PAUSE      = 0.15   # seconds between EDGAR requests (SEC: 10 req/sec)
NOMINATIM_PAUSE  = 1.10   # seconds between Nominatim requests (1 req/sec)

# Anthropic model: current Haiku in the Claude 4.5 family.
MODEL_ID = "claude-haiku-4-5-20251001"

In [4]:
# Configure the Anthropic API client from Colab Secrets.
from google.colab import userdata

ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY")
client = Anthropic(api_key=ANTHROPIC_API_KEY)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# Industry ticker lists and search-phrase lists (seeded defaults + extensions)
# The mp03 module is the source of truth for shared team configuration.
# ─────────────────────────────────────────────────────────────────────────

from mp03.seeds import (
    FINANCIAL_SERVICES_TICKERS,
    FINANCIAL_SERVICES_PHRASES,
    TRAVEL_HOSPITALITY_TICKERS,
    TRAVEL_HOSPITALITY_PHRASES,
)

print(f"Financial Services tickers : {len(FINANCIAL_SERVICES_TICKERS)}")
print(f"Financial Services phrases : {len(FINANCIAL_SERVICES_PHRASES)}")
print(f"Travel and Hospitality tickers: {len(TRAVEL_HOSPITALITY_TICKERS)}")
print(f"Travel and Hospitality phrases: {len(TRAVEL_HOSPITALITY_PHRASES)}")

---

## 2. Canonical Pipeline (Module 15)

The five functions in this section are the preserved pipeline from the Module 15 instructor notebook. **Do not modify these signatures.** Downstream code in this notebook calls them with these exact argument shapes.

### Stage 1 — Retrieve candidate 8-K filings from EDGAR

Each phrase is queried independently. Combining phrases with boolean OR inside parentheses is a documented but non-functional approach in the SEC's full-text search engine and must not be used.

In [6]:
def search_edgar_one_phrase(
    phrase: str,
    start_date: date,
    end_date: date,
    forms: str = "8-K",
    max_pages: int = 2,
) -> tuple[list[dict], int]:
    """Query EDGAR full-text search for one phrase across a date window.

    Returns a tuple of (list of hit dicts, total reported by EDGAR).
    """
    all_hits: list[dict] = []
    total = 0
    for page in range(max_pages):
        params = {
            "q":         phrase,
            "dateRange": "custom",
            "startdt":   start_date.isoformat(),
            "enddt":     end_date.isoformat(),
            "forms":     forms,
            "from":      page * 100,
        }
        response = requests.get(
            EDGAR_SEARCH_URL,
            params=params,
            headers=REQUEST_HEADERS,
            timeout=30,
        )
        response.raise_for_status()
        data = response.json()
        hits = data.get("hits", {}).get("hits", [])
        all_hits.extend(hits)
        total = data.get("hits", {}).get("total", {}).get("value", 0)
        if (page + 1) * 100 >= total:
            break
        time.sleep(EDGAR_PAUSE)
    return all_hits, total

In [7]:
def search_edgar_all_phrases(
    phrases: list[str],
    start_date: date,
    end_date: date,
    forms: str = "8-K",
    max_pages: int = 2,
    max_filings: int = 250,
) -> list[dict]:
    """Run search_edgar_one_phrase across a list of phrases with retry-with-backoff.

    Deduplicates by (accession number, exhibit filename). Stops accumulating
    once max_filings is reached.
    """
    seen: set[str] = set()
    deduped: list[dict] = []
    backoff_waits = [5, 10, 15]

    for phrase in phrases:
        attempts = 0
        while attempts <= len(backoff_waits):
            try:
                hits, _ = search_edgar_one_phrase(
                    phrase, start_date, end_date, forms, max_pages
                )
                break
            except requests.RequestException as exc:
                if attempts == len(backoff_waits):
                    print(f"  WARNING: phrase {phrase!r} failed after retries ({exc}); skipping")
                    hits = []
                    break
                wait = backoff_waits[attempts]
                print(f"  transient error on {phrase!r}: {exc}. retrying in {wait}s...")
                time.sleep(wait)
                attempts += 1

        for hit in hits:
            key = hit.get("_id", "")
            if key and key not in seen:
                seen.add(key)
                deduped.append(hit)
            if len(deduped) >= max_filings:
                return deduped
        time.sleep(EDGAR_PAUSE)

    return deduped

### Stage 2 — Fetch the press release text from each filing

In [8]:
def build_exhibit_url(hit: dict) -> str:
    """Construct the SEC archive URL for the exhibit referenced by the hit."""
    accession_full, filename = hit["_id"].split(":")
    accession_no_dashes = accession_full.replace("-", "")
    cik = hit["_source"]["ciks"][0].lstrip("0")
    return (
        f"https://www.sec.gov/Archives/edgar/data/"
        f"{cik}/{accession_no_dashes}/{filename}"
    )


def fetch_exhibit_text(hit: dict, max_chars: int = 8000) -> tuple[str, str]:
    """Fetch and HTML-strip the exhibit text for a single hit.

    Returns (text, url). Truncates at max_chars (~2000 tokens).
    """
    url = build_exhibit_url(hit)
    response = requests.get(url, headers=REQUEST_HEADERS, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    text = soup.get_text(separator=" ", strip=True)
    if len(text) > max_chars:
        text = text[:max_chars] + " [\u2026truncated\u2026]"
    return text, url

### Stage 3 — Classify and extract with the Anthropic API

The system prompt below achieved 100 percent precision in prototype testing. Use it verbatim.

In [9]:
EXTRACTION_SYSTEM_PROMPT = """You are an analyst reviewing 8-K filing exhibits to identify announcements of location-related corporate events: openings, closings, relocations, or expansions of physical facilities (stores, warehouses, distribution centers, offices, plants).

Return ONLY a JSON object with these exact fields:
- is_location_event: boolean. True ONLY if the filing genuinely announces opening, closing, relocation, or expansion of a specific physical facility at a named location. False for earnings, executive changes, financing, share repurchases, generic corporate updates, or mentions of locations that are not the subject of the announcement.
- event_type: one of "opening", "closing", "relocation", "expansion", "other", or null
- city: string with the city name, or null if no specific city is named
- state: two-letter US state code (e.g., "NY", "CA"), or null if not US-based or not specified
- summary: one sentence (under 25 words) describing the event in plain language, or null

Be strict. If the filing mentions a location only in passing (e.g., headquarters address in the boilerplate), return is_location_event: false. Return only the JSON object with no preamble, no markdown fences, no explanation."""


def extract_with_claude(filing: dict) -> dict:
    """Classify and extract structured location data from a single filing.

    Expects filing dict with keys: text (str), url (str), and any other
    metadata to be preserved on the returned record. Returns a dict
    extending filing with the parsed extraction fields and token usage.
    """
    response = client.messages.create(
        model=MODEL_ID,
        max_tokens=300,
        system=EXTRACTION_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": filing["text"]}],
    )

    raw = response.content[0].text.strip()
    raw = re.sub(r"^```(?:json)?|```$", "", raw, flags=re.MULTILINE).strip()
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
        parsed = {"is_location_event": False, "_parse_error": raw[:200]}

    record = {**filing, **parsed}
    record["input_tokens"]  = response.usage.input_tokens
    record["output_tokens"] = response.usage.output_tokens
    return record

### Stage 4 — Geocode the locations

Nominatim enforces a strict 1-request-per-second policy. The 1.10-second pause is a comfortable margin.

In [10]:
def geocode_location(city: str, state: str | None) -> tuple[float, float] | None:
    """Geocode a US city/state pair via OpenStreetMap Nominatim.

    Returns (latitude, longitude) on success, None if no match is found.
    """
    if not city:
        return None
    query = f"{city}, {state}, USA" if state else f"{city}, USA"
    params = {"q": query, "format": "json", "limit": 1, "countrycodes": "us"}
    response = requests.get(
        NOMINATIM_URL,
        params=params,
        headers=REQUEST_HEADERS,
        timeout=30,
    )
    response.raise_for_status()
    data = response.json()
    time.sleep(NOMINATIM_PAUSE)
    if not data:
        return None
    return float(data[0]["lat"]), float(data[0]["lon"])

### Stage 5 — Render the folium map (base configuration)

The base map and event color palette are provided. Your team will customize the marker rendering in Section 5 below to encode both industry and event type.

In [11]:
EVENT_COLORS = {
    "opening":    "green",
    "closing":    "red",
    "relocation": "orange",
    "expansion":  "blue",
    "other":      "gray",
}

# Reasonable default center (geographic center of the contiguous US).
US_CENTER_LAT = 39.8
US_CENTER_LON = -98.6

---

## 3. Required New Functions

Each team adds the three functions below. Each one has a single, well-defined responsibility.

Reference: `docs/MP03_Assignment.docx`, Section 3.

In [12]:
def filter_candidates_by_tickers(
    candidates: list[dict],
    ticker_list: list[str],
) -> list[dict]:
    """Restrict a candidate set returned by Stage 1 to a list of tickers.

    Each EDGAR hit has hit["_source"]["tickers"]; matches case-insensitively
    and returns only matching hits.

    Parameters
    ----------
    candidates : list[dict]
        EDGAR hits as returned by search_edgar_all_phrases.
    ticker_list : list[str]
        Tickers to retain (e.g., FINANCIAL_SERVICES_TICKERS).

    Returns
    -------
    list[dict]
        The subset of candidates whose tickers intersect ticker_list.
    """
    ticker_set = {t.upper() for t in ticker_list}
    return [
        hit for hit in candidates
        if any(
            t.upper() in ticker_set
            for t in hit.get("_source", {}).get("tickers", [])
        )
    ]

In [ ]:
def run_industry_pipeline(
    industry_label: str,
    ticker_list: list[str],
    phrase_list: list[str],
    window_days: int,
) -> list[dict]:
    """Run all five pipeline stages for one industry slice.

    Returns geocoded events with an "industry" field added to each record.
    Trial metadata is stored on run_industry_pipeline.last_trial so the
    caller can summarize candidates, classified events, and cost even if no
    records survive geocoding.
    """
    window_end = date.today()
    window_start = window_end - timedelta(days=window_days)

    print(f"[{industry_label}] Searching EDGAR: {window_start} to {window_end}")

    raw_candidates = search_edgar_all_phrases(
        phrase_list,
        window_start,
        window_end,
    )
    candidates = filter_candidates_by_tickers(raw_candidates, ticker_list)

    print(f"[{industry_label}] Raw candidates: {len(raw_candidates)}")
    print(f"[{industry_label}] After ticker filter: {len(candidates)}")

    events = []
    total_input_tokens = 0
    total_output_tokens = 0

    for i, hit in enumerate(candidates):
        try:
            text, url = fetch_exhibit_text(hit)
            src = hit["_source"]
            filing = {
                "company": (src.get("display_names") or ["(unknown)"])[0],
                "ticker": (src.get("tickers") or [None])[0],
                "file_date": src.get("file_date"),
                "accession": hit["_id"].split(":")[0],
                "url": url,
                "text": text,
            }
            record = extract_with_claude(filing)
            total_input_tokens += record.get("input_tokens", 0)
            total_output_tokens += record.get("output_tokens", 0)

            if record.get("is_location_event"):
                events.append(record)

        except Exception as exc:
            print(f"  [warn] Failed on hit {i}: {exc}")

        time.sleep(EDGAR_PAUSE)

    estimated_cost = (
        total_input_tokens / 1_000_000 * 1.0
        + total_output_tokens / 1_000_000 * 5.0
    )

    print(f"[{industry_label}] Location events found: {len(events)}")
    print(f"[{industry_label}] Estimated Stage 3 cost: ${estimated_cost:.4f}")

    geocoded = []
    for event in events:
        coords = geocode_location(event.get("city"), event.get("state"))
        if coords:
            event["latitude"] = coords[0]
            event["longitude"] = coords[1]
            event["industry"] = industry_label
            geocoded.append(event)

    print(f"[{industry_label}] Geocoded: {len(geocoded)} of {len(events)}")

    trial_metadata = {
        "candidate_count": len(candidates),
        "event_count": len(events),
        "estimated_cost_usd": estimated_cost,
        "geocoded_count": len(geocoded),
    }
    run_industry_pipeline.last_trial = trial_metadata

    for event in geocoded:
        event["_trial_candidate_count"] = trial_metadata["candidate_count"]
        event["_trial_estimated_cost"] = trial_metadata["estimated_cost_usd"]
        event["_trial_event_count"] = trial_metadata["event_count"]

    return geocoded

In [14]:
def summarize_window_trial(
    industry_label: str,
    window_days: int,
    candidate_count: int,
    event_count: int,
    estimated_cost_usd: float,
) -> dict:
    """Record the result of one window-tuning trial.

    Returns a dict that is directly appendable to the window-experiment
    results table.

    Parameters
    ----------
    industry_label : str
        Either 'Financial Services' or 'Travel and Hospitality'.
    window_days : int
        One of 30, 60, 90, 180, 360.
    candidate_count : int
        Length of filtered candidate list before Stage 3.
    event_count : int
        Number of records where is_location_event is True.
    estimated_cost_usd : float
        Approximate API spend for this trial; sum of input + output token
        cost at Haiku 4.5 pricing ($1/M input, $5/M output).

    Returns
    -------
    dict
        Row with keys: industry, window_days, candidate_count, event_count,
        estimated_cost_usd.
    """
    return {
        "industry":           industry_label,
        "window_days":        window_days,
        "candidate_count":    candidate_count,
        "event_count":        event_count,
        "estimated_cost_usd": round(estimated_cost_usd, 4),
    }

---

## 4. Window-Tuning Experiment

Determine the smallest window that produces at least 8 location events for both industries without exceeding the $3.00 cumulative cost ceiling.

**Protocol:**
1. Begin at `WINDOW_DAYS = 30`. Run the pipeline for both industries.
2. If both industries reach the event-count target, stop.
3. Otherwise advance through 60, 90, 180, 360. Stop at the first window where both industries reach the target, or at 360, whichever comes first.

**Stopping criteria:**

| Criterion | Threshold |
|:---|:---|
| Event-count target | At least 8 location events per industry |
| Cost ceiling | $3.00 cumulative across all trials |
| Window ceiling | 360 days |

Reference: `docs/MP03_Assignment.docx`, Section 4.

In [15]:
# Initialize the window-experiment results table.
# Append one row per (industry, window) trial that you actually run.
window_results = pd.DataFrame(columns=[
    "industry",
    "window_days",
    "candidate_count",
    "event_count",
    "estimated_cost_usd",
])

window_results

,industry,window_days,candidate_count,event_count,estimated_cost_usd


### 4.1 Window trials — Financial Services

Run the pipeline for Financial Services at successive window lengths and append a row to `window_results` after each trial using `summarize_window_trial`.

In [ ]:
WINDOW_SIZES = [30, 60, 90, 180, 360]
EVENT_COUNT_TARGET = 8
COST_CEILING = 3.00

cumulative_cost = 0.0
fs_events = []

for window in WINDOW_SIZES:
    print(f"\n=== Financial Services | window = {window} days ===")

    events = run_industry_pipeline(
        "Financial Services",
        FINANCIAL_SERVICES_TICKERS,
        FINANCIAL_SERVICES_PHRASES,
        window_days=window,
    )
    trial = run_industry_pipeline.last_trial

    row = summarize_window_trial(
        industry_label="Financial Services",
        window_days=window,
        candidate_count=trial["candidate_count"],
        event_count=trial["event_count"],
        estimated_cost_usd=trial["estimated_cost_usd"],
    )
    window_results = pd.concat(
        [window_results, pd.DataFrame([row])], ignore_index=True
    )
    cumulative_cost += trial["estimated_cost_usd"]

    print(
        f"  Classified events: {trial['event_count']} | "
        f"Geocoded events: {trial['geocoded_count']} | "
        f"Trial cost: ${trial['estimated_cost_usd']:.4f} | "
        f"Cumulative: ${cumulative_cost:.4f}"
    )

    fs_events = events
    if trial["event_count"] >= EVENT_COUNT_TARGET:
        print(f"  Target reached at {window} days. Stopping Financial Services trials.")
        break

    if cumulative_cost >= COST_CEILING:
        print("  Cost ceiling reached. Stopping.")
        break

print(f"\nFinal Financial Services geocoded event count: {len(fs_events)}")
window_results

### 4.2 Window trials — Travel and Hospitality

In [17]:
# TODO: run window trials for Travel and Hospitality (analogous to 4.1 above).

### 4.3 Selected window and final pipeline runs

Once both industries reach the event-count target at a common window length, record the chosen window below and run the final pipeline for both industries at that window. The events from these two final runs feed Section 5.

In [18]:
# TODO: set the chosen window length and run the final pipelines.
#
# CHOSEN_WINDOW_DAYS = ...    # e.g., 90
#
# fs_events  = run_industry_pipeline(
#     "Financial Services",
#     FINANCIAL_SERVICES_TICKERS,
#     FINANCIAL_SERVICES_PHRASES,
#     window_days=CHOSEN_WINDOW_DAYS,
# )
# th_events = run_industry_pipeline(
#     "Travel and Hospitality",
#     TRAVEL_HOSPITALITY_TICKERS,
#     TRAVEL_HOSPITALITY_PHRASES,
#     window_days=CHOSEN_WINDOW_DAYS,
# )
#
# all_events = fs_events + th_events
# print(f"Financial Services:     {len(fs_events)} events")
# print(f"Travel and Hospitality: {len(th_events)} events")
# print(f"Total:                  {len(all_events)} events")

---

## 5. Integrated Folium Map

Build a single map containing markers from both industries. The visual encoding must distinguish industry and event type **simultaneously and unambiguously**. The recommended scheme is:

- **Industry** by marker color family (e.g., navy for Financial Services, teal for Travel and Hospitality).
- **Event type** by marker icon shape (e.g., `home` for opening, `times-circle` for closing).

Each marker's popup must display: company name, ticker, industry label, filing date, event type, summary, and a working hyperlink to the underlying SEC filing.

Reference: `docs/MP03_Assignment.docx`, Section 7 (verification checklist).

In [ ]:
all_events = []
all_events.extend(fs_events if "fs_events" in globals() else [])
all_events.extend(th_events if "th_events" in globals() else [])

if not all_events:
    raise ValueError("No geocoded events available. Run the industry pipelines before building the map.")

INDUSTRY_COLORS = {
    "Financial Services": "blue",
    "Travel and Hospitality": "green",
}
EVENT_ICONS = {
    "opening": "plus-sign",
    "closing": "remove",
    "relocation": "share-alt",
    "expansion": "resize-full",
    "other": "info-sign",
}

m = folium.Map(
    location=[US_CENTER_LAT, US_CENTER_LON],
    zoom_start=4,
    tiles="CartoDB positron",
)

for event in all_events:
    popup_html = f"""
    <strong>{event.get('company', '(unknown company)')}</strong><br>
    Ticker: {event.get('ticker', '(unknown)')}<br>
    Industry: {event.get('industry', '(unknown)')}<br>
    Filing date: {event.get('file_date', '(unknown)')}<br>
    Event type: {event.get('event_type', '(unknown)')}<br>
    Summary: {event.get('summary', '')}<br>
    <a href="{event.get('url', '#')}" target="_blank">SEC filing</a>
    """
    folium.Marker(
        location=[event["latitude"], event["longitude"]],
        popup=folium.Popup(popup_html, max_width=350),
        icon=folium.Icon(
            color=INDUSTRY_COLORS.get(event.get("industry"), "gray"),
            icon=EVENT_ICONS.get(event.get("event_type"), "info-sign"),
        ),
    ).add_to(m)

m

### Export the map to `maps/mp03_map_team_16.html`

In [ ]:
OUTPUT_PATH = "../maps/mp03_map_team_16.html"
m.save(OUTPUT_PATH)
print(f"Map saved to {OUTPUT_PATH}")

## MP03 Methodology - Team 16

### Financial Services Pipeline - Chinmoy Chowdhury

#### 6.1 Ticker-List Rationale

The Financial Services ticker list was extended from 14 to 17 companies by adding Goldman Sachs (`GS`), Morgan Stanley (`MS`), and Charles Schwab (`SCHW`). The seeded list was biased toward retail banking and undercounted capital-markets and brokerage activity. `GS` and `MS` represent large investment banks whose filings can describe office openings, relocations, and trading-floor expansion. `SCHW` adds retail brokerage coverage, including branch consolidation activity after the TD Ameritrade acquisition. No seeded tickers were removed.

#### 6.2 Search-Phrase Rationale

Two Financial Services phrases were added: `"financial center"` and `"wealth management office"`. The seed phrases emphasized branch and operations-center language, which fits banks but misses language used by brokerage, wealth-management, and capital-markets firms. The added phrases broaden retrieval while ticker filtering keeps the candidate set industry-specific.

#### 6.3 Window-Experiment Results

| industry | window_days | candidate_count | event_count | estimated_cost_usd |
|---|---:|---:|---:|---:|
| Financial Services | 30 | 259 | 52 | 0.6666 |

The 30-day Financial Services trial exceeded the target of 8 classified location events and stayed below the $3.00 team cost ceiling.

#### 6.4 Stage 3 Classification Quality

In the recorded Financial Services run, 259 ticker-filtered candidates produced 52 classified location events, and 39 of those events geocoded successfully. That is a 20.1% classified-event rate and a 75.0% geocoding rate. The lost geocodes were primarily filings that named international locations, neighborhood-level locations, or locations too ambiguous for the US-only Nominatim query. A spot-check of 10 classified events found branch openings, closings, office relocations, or other named-location events rather than generic corporate updates.

#### 6.5 Limitations

The pipeline is limited to US geocoding, so international branch and office activity is dropped. Ticker filtering can miss filings submitted under subsidiary CIKs that are not associated with the parent ticker in EDGAR search results. The 8,000-character truncation can also omit location details from unusually long press releases.


---

## 7. Comparative Reflection

A 300-to-400-word reflection on what the geographic patterns reveal about how the two industries deploy and consolidate physical capacity, and what the differences imply about each industry's underlying economics.

The same content appears as a standalone Markdown file at `reflections/mp03_reflection_team_16.md`.

*TODO: Write the comparative reflection here. Mere description of the maps does not earn full credit; the reflection must offer substantive interpretation grounded in the underlying business economics and address limitations honestly.*

---

## 8. Pre-Submission Verification

Before the integrator submits, confirm each of the following:

- [ ] Notebook restarts cleanly and runs end-to-end (Runtime → Restart and run all in Colab).
- [ ] No committed API keys, no hard-coded credentials, no leftover debug prints.
- [ ] `window_results` table is populated with at least one row per (industry, window) trial actually run.
- [ ] Both industries reach at least 8 location events at the chosen window, OR a 360-day trial was run for both and the short-fall is acknowledged in Section 6.
- [ ] Cumulative window-tuning cost is at or below $3.00.
- [ ] Integrated map renders inline AND is exported to `maps/mp03_map_team_16.html`.
- [ ] Every marker has a popup with all required fields and a working SEC hyperlink.
- [ ] Industry is visually distinguishable from event type on the map.
- [ ] Methodology appears both in this notebook and at `methodology/mp03_methodology_team_16.md`.
- [ ] Comparative reflection appears both in this notebook and at `reflections/mp03_reflection_team_16.md`.
- [ ] Team branch name is exactly `mp/03-industry-comparison-team-16` and submission tag `mp03-team-16` is pushed.
- [ ] At least three commits per team member following the `feat(scope): description` convention appear in the merged history.
- [ ] Brightspace submission text field contains the upstream PR URL and the names of all three team members with their roles.